[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HesusG/dammg-supervisado/blob/main/notebooks/demos/S1_polinomial.ipynb)

# Demo: cuando la recta no alcanza (regresión polinomial)

Demo en vivo de S1, unos 10 minutos. No se entrega. Somos el equipo de análisis de MiBici y nos toca una pregunta concreta
de la estación 51. Primero la intentamos con una recta, vemos dónde falla y luego le damos curva.
Corre las celdas en orden; las dos primeras preparan el entorno hasta que aparezca `✅ entorno listo`.

In [ ]:
# @title Preparación del entorno: solo córrela, no hace falta leerla
# Descarga los datos y dammg_utils.py, revisa que no lleguen dañados y avisa con ✅ entorno listo.
import hashlib, importlib.util, os, pathlib, subprocess, sys, urllib.request

BASE_URL = "https://hesusg.github.io/dammg-supervisado/datos/"
# Versiones fijas: copia generada de notebooks/requirements.txt (fuente única)
REQUISITOS = {
    'numpy': 'numpy==2.5.4',
    'pandas': 'pandas==3.0.6',
    'sklearn': 'scikit-learn==1.9.1',
    'matplotlib': 'matplotlib==3.11.2',
    'ipywidgets': 'ipywidgets==8.1.9',
    'pyarrow': 'pyarrow==26.0.0',
}
SHA256 = {
    'mibici_curso.parquet': 'a5fd844c57bbf220f0ddfc27dc965ef8fb8438bd6c1167709bde7ccc1dafa6d8',
    'dammg_utils.py': '7499b8695cb94d4dd72133e197d95a143e8b05b82f1d91928f6a8a30fa0f62d0',
}
EN_EVALUADOR = os.environ.get("DAMMG_GRADER") == "1"
# Umbral del evento |future_net| (percentil 85 de train), congelado para todo el curso
UMBRAL_EVENTO = 7.0


def _sha256(ruta):
    return hashlib.sha256(pathlib.Path(ruta).read_bytes()).hexdigest()


def _copia_local(nombre):
    """Busca una copia local: DAMMG_DATA_DIR o el repo del curso (data/curso, notebooks)."""
    carpetas = []
    if os.environ.get("DAMMG_DATA_DIR"):
        carpetas.append(pathlib.Path(os.environ["DAMMG_DATA_DIR"]))
    aqui = pathlib.Path.cwd().resolve()
    for p in [aqui, *aqui.parents][:5]:
        carpetas += [p / "data" / "curso", p / "notebooks", p]
    for c in carpetas:
        if (c / nombre).is_file():
            return c / nombre
    return None


def _obtener(nombre, esperado):
    """Copia local si existe; si no, descarga del sitio del curso y verifica el sha256."""
    if EN_EVALUADOR:
        # Dentro del evaluador: nada de descargar; los datos ya están en /data
        return pathlib.Path("/data") / nombre
    local = _copia_local(nombre)
    if local is not None:
        if _sha256(local) != esperado:
            print(f"⚠ {nombre}: la copia local no coincide con el sha256 publicado (¿versión en desarrollo?)")
        return local
    destino = pathlib.Path("datos_dammg") / nombre
    destino.parent.mkdir(exist_ok=True)
    if not destino.exists() or _sha256(destino) != esperado:
        print(f"Descargando {nombre} ...")
        try:
            urllib.request.urlretrieve(BASE_URL + nombre, destino)
        except OSError as e:   # HTTPError, sin internet, etc.
            raise RuntimeError(
                f"No pude descargar {nombre} ({e}). Revisa tu conexión y vuelve a ejecutar esta celda. "
                "Si sigue fallando, avisa en el foro de Moodle: puede que el archivo aún no esté publicado."
            ) from None
    assert _sha256(destino) == esperado, f"{nombre} llegó incompleto o alterado: bórralo y vuelve a ejecutar."
    return destino


if not EN_EVALUADOR:
    faltan = [pin for modulo, pin in REQUISITOS.items() if importlib.util.find_spec(modulo) is None]
    if faltan:
        print("Instalando:", ", ".join(faltan))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltan])
RUTAS = {nombre: _obtener(nombre, esperado) for nombre, esperado in SHA256.items()}

sys.path.insert(0, str(RUTAS["dammg_utils.py"].parent))
RUTA_DATOS = RUTAS["mibici_curso.parquet"]

import dammg_utils as du
du.entorno_listo()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dammg_utils import COLORES
from IPython.display import display
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, PolynomialFeatures
du.estilo_dammg()
np.set_printoptions(precision=4, suppress=True)

## 1. La pregunta de negocio

**¿Cuántas bicis salen a cada hora de la estación 51 entre semana?** Con esa respuesta el equipo de redistribución
decide a qué hora llevar bicis y a qué hora sobran.

Usamos una sola variable, la hora del día (`hour_of_day`, de 0 a 23), y la $y$ son las salidas (`departures`).
Ajustamos con `train` (octubre a junio) y validamos con julio, que el modelo no ve.

In [ ]:
df = pd.read_parquet(RUTA_DATOS)
e51 = df[(df.station_id == 51) & (df.is_weekend == 0)]      # estación 51, de lunes a viernes
train = e51[e51.particion == "train"]                        # octubre a junio: aquí se ajusta
julio = e51[e51.particion == "val"]                          # julio: aquí se valida

X_train, y_train = train[["hour_of_day"]], train.departures
X_julio, y_julio = julio[["hour_of_day"]], julio.departures
print(f"train: {len(train):,} horas   julio: {len(julio):,} horas")

prom = train.groupby("hour_of_day").departures.mean()
display(prom.round(1).to_frame("salidas promedio en train").T)

**¿Qué esperamos ver?** Unas 4,500 horas de train y unas 550 de julio. En la tabla, casi cero en la madrugada,
un pico fuerte a las 7 y 8 a.m. y otro más bajo hacia las 6 p.m.

In [ ]:
rng = np.random.default_rng(0)
brinco = rng.uniform(-0.3, 0.3, len(train))     # separa un poco los puntos de la misma hora para que se vean

plt.figure(figsize=(9, 4))
plt.scatter(train.hour_of_day + brinco, train.departures, s=6, alpha=0.15, color=COLORES["w"],
            edgecolors="none", label="una hora de un día (train)")
plt.plot(prom.index, prom.values, color=COLORES["tinta"], marker="o", label="promedio de cada hora")
plt.xlabel("hora del día")
plt.ylabel("salidas")
plt.xticks(range(0, 24, 2))
plt.title("Estación 51, entre semana, octubre a junio")
plt.legend(loc="upper left")
plt.show()

**¿Qué esperamos ver?** Una nube que sube de golpe en la mañana, baja a mediodía, vuelve a subir en la tarde y cae en la noche.
El promedio dibuja dos jorobas. Eso no es una recta.

## 2. Lo intentamos con una recta

$$\hat y = w\,x + b$$

En palabras: las salidas que predecimos son un peso $w$ por la hora $x$, más un número fijo $b$. Cada hora suma lo mismo, $w$ bicis.

**¿Por qué empezamos por aquí?** Porque es lo más simple y ya sabemos ajustarla. Si la recta alcanza, no hace falta nada más.

In [ ]:
recta = LinearRegression().fit(X_train, y_train)
print(f"ŷ = {recta.coef_[0]:.2f} x + {recta.intercept_:.2f}")


def metricas(modelo, nombre, Xtr=None, ytr=None):
    """MAE y R² en train y en julio."""
    Xtr = X_train if Xtr is None else Xtr
    ytr = y_train if ytr is None else ytr
    return {"modelo": nombre,
            "MAE train": mean_absolute_error(ytr, modelo.predict(Xtr)),
            "MAE julio": mean_absolute_error(y_julio, modelo.predict(X_julio)),
            "R² train": r2_score(ytr, modelo.predict(Xtr)),
            "R² julio": r2_score(y_julio, modelo.predict(X_julio))}


display(pd.DataFrame([metricas(recta, "recta")]).round(3))

horas = np.linspace(0, 23, 300)
rejilla = pd.DataFrame({"hour_of_day": horas})
plt.figure(figsize=(9, 4))
plt.scatter(train.hour_of_day + brinco, train.departures, s=6, alpha=0.15, color=COLORES["w"], edgecolors="none")
plt.plot(prom.index, prom.values, color=COLORES["tinta"], marker="o", label="promedio de cada hora")
plt.plot(horas, recta.predict(rejilla), color=COLORES["error"], lw=3.5, label="la recta")
plt.xlabel("hora del día")
plt.ylabel("salidas")
plt.xticks(range(0, 24, 2))
plt.legend(loc="upper left")
plt.show()

**¿Qué esperamos ver?** Una recta casi plana que sube poco a lo largo del día y pasa por en medio de las dos jorobas.
El MAE queda alrededor de 7 bicis en train y 8.5 en julio, y el $R^2$ casi en cero: la recta explica casi nada más que el promedio.

### ¿Dónde se equivoca?

Para cada hora promediamos el error $\hat y - y$:

$$\text{error de la hora } h = \frac{1}{m_h}\sum_{i:\ x_i = h} (\hat y_i - y_i)$$

En palabras: tomamos todas las filas de esa hora, restamos lo real a lo predicho y sacamos el promedio.
Positivo quiere decir que la recta **predice de más**; negativo, que **se queda corta**.

**¿Por qué por hora?** Si los errores fueran ruido, cada hora tendría un promedio cercano a cero y sin orden.
Si tienen forma, al modelo le falta algo que sí se puede aprender.

In [ ]:
def errores_por_hora(modelo, datos, X):
    return (modelo.predict(X) - datos.departures).groupby(datos.hour_of_day).mean()


err_recta = pd.DataFrame({"train": errores_por_hora(recta, train, X_train),
                          "julio": errores_por_hora(recta, julio, X_julio)})
display(err_recta.loc[[2, 4, 7, 8, 13, 18, 23]].round(1).T)

plt.figure(figsize=(9, 3.8))
colores = [COLORES["error"] if v < 0 else COLORES["w"] for v in err_recta.train]
plt.bar(err_recta.index, err_recta.train, color=colores, edgecolor=COLORES["tinta"], label="train")
plt.plot(err_recta.index, err_recta.julio, color=COLORES["tinta"], marker="o", lw=1.5, label="julio")
plt.axhline(0, color=COLORES["tinta"], lw=1.5)
plt.xlabel("hora del día")
plt.ylabel("error promedio ŷ − y (bicis)")
plt.xticks(range(0, 24, 2))
plt.title("La recta: azul = predice de más, rojo = se queda corta")
plt.legend()
plt.show()
print(f"Madrugada (1 a 4 a.m.): la recta predice {err_recta.train.loc[1:4].mean():+.1f} bicis de más en promedio")
print(f"7 y 8 a.m.: se queda corta por {-err_recta.train.loc[7:8].mean():.1f} bicis en promedio")

**¿Qué esperamos ver?** Errores con forma, no ruido. En la madrugada la recta predice unas 7 u 8 bicis cuando no sale ninguna (barras azules);
a las 7 y 8 a.m. se queda corta por casi 20 (barras rojas); en la noche vuelve a pasarse. Julio repite el mismo dibujo.

**Diagnóstico:** a la recta le falta curva. No es mala suerte de los datos, es la forma del modelo.

## 3. Por qué polinomial

El patrón es **curvo y suave**, y depende de **una sola variable**. Para eso sirve un polinomio:

$$\hat y = w_1 x + w_2 x^2 + \dots + w_d x^d + b$$

En palabras: a la hora le agregamos sus potencias $x^2, x^3, \dots, x^d$ como columnas nuevas, y a cada una le buscamos un peso.
El **grado** $d$ dice cuántas potencias usamos. Con $d = 1$ es la recta de antes.

**¿Por qué sigue siendo regresión lineal?** Porque el modelo es una suma de columnas por pesos. Que las columnas vengan de la misma $x$
no lo sabe ni le importa. `PolynomialFeatures` arma esas columnas; `LinearRegression` hace lo de siempre.

In [ ]:
unas_horas = pd.DataFrame({"hour_of_day": [0, 2, 6, 8, 12, 18, 23]})
poli3 = PolynomialFeatures(degree=3, include_bias=False)
columnas = pd.DataFrame(poli3.fit_transform(unas_horas), columns=["x", "x²", "x³"]).astype(int)
display(columnas)

**¿Qué esperamos ver?** Una fila por hora con $x$, $x^2$ y $x^3$. Con $x = 2$ sale 2, 4, 8, como en la slide. Con $x = 23$ ya sale 12,167.

### Por qué escalamos la hora a $[-1, 1]$

$$x' = -1 + 2\,\frac{x - 0}{23 - 0}$$

En palabras: la hora 0 pasa a $-1$, la 23 pasa a $1$ y las demás quedan en medio, en línea recta.

Con la hora tal cual, $23^8$ es casi 80 mil millones, mientras $1^8$ es 1. Las columnas viven en escalas tan distintas que la computadora
pierde precisión al resolver. Con la hora escalada, cualquier potencia de un número entre $-1$ y $1$ se queda entre $-1$ y $1$.

In [ ]:
escala = MinMaxScaler(feature_range=(-1, 1)).fit(X_train)    # aprende el 0 y el 23 de train
x_raw = unas_horas.hour_of_day.to_numpy(dtype=float)
x_esc = escala.transform(unas_horas).ravel()
tabla = pd.DataFrame({"x": x_raw, "x⁸": x_raw ** 8, "x' (escalada)": x_esc, "x'⁸": x_esc ** 8})
tabla["x⁸"] = tabla["x⁸"].map("{:,.0f}".format)     # con comas, para leer el tamaño
display(tabla.round(4))

potencias = np.arange(1, 13)
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar(potencias - 0.2, [23.0 ** k for k in potencias], width=0.4, color=COLORES["error"], edgecolor=COLORES["tinta"], label="hora tal cual (x = 23)")
ax.bar(potencias + 0.2, [1.0 ** k for k in potencias], width=0.4, color=COLORES["w"], edgecolor=COLORES["tinta"], label="hora escalada (x' = 1)")
ax.set(yscale="log", xlabel="potencia k", ylabel="valor más grande de la columna xᵏ", xticks=potencias,
       title="La columna más grande de cada potencia")
ax.legend()
plt.show()

**¿Qué esperamos ver?** En la tabla, $x^8$ de las 23 horas da 78,310,985,281; escalada da 1. En la gráfica (escala logarítmica) las barras rojas
crecen sin parar hasta $10^{16}$ y las azules se quedan en 1.

¿Eso importa en la práctica? Ajustemos el grado 8 de las dos formas con los mismos datos.

In [ ]:
def ajusta(grado, X, y, escalar=True):
    pasos = [MinMaxScaler(feature_range=(-1, 1))] if escalar else []
    return make_pipeline(*pasos, PolynomialFeatures(grado), LinearRegression()).fit(X, y)


comparar = pd.DataFrame([
    metricas(ajusta(2, X_train, y_train), "grado 2, escalada"),
    metricas(ajusta(8, X_train, y_train, escalar=False), "grado 8, hora tal cual"),
    metricas(ajusta(8, X_train, y_train), "grado 8, escalada"),
]).round(3)
display(comparar)

**¿Qué esperamos ver?** El grado 8 sin escalar sale **peor en train que el grado 2** (MAE de 6.5 contra 5.7). En matemáticas eso no puede pasar:
el grado 8 puede copiar al grado 2 poniendo ceros en las demás potencias. Es la computadora que se ahoga con números enormes.
Escalado, el grado 8 baja a 4.5. Desde aquí siempre escalamos: `make_pipeline(MinMaxScaler((-1, 1)), PolynomialFeatures(d), LinearRegression())`.

## 4. ¿De qué grado?

Primero lo honesto: con todos los datos de train, ¿qué le pasa al MAE cuando subimos el grado de 1 a 12?

In [ ]:
grados = range(1, 13)
curva_todo = pd.DataFrame([metricas(ajusta(g, X_train, y_train), g) for g in grados]).set_index("modelo")
curva_todo.index.name = "grado"
display(curva_todo[["MAE train", "MAE julio"]].round(2).T)

plt.figure(figsize=(8, 3.6))
plt.plot(grados, curva_todo["MAE train"], marker="o", label="train")
plt.plot(grados, curva_todo["MAE julio"], marker="o", color=COLORES["error"], label="julio")
plt.xlabel("grado")
plt.ylabel("MAE (bicis por hora)")
plt.xticks(list(grados))
plt.title(f"Con todo train ({len(train):,} horas)")
plt.legend()
plt.show()

**¿Qué esperamos ver?** Las dos curvas bajan casi juntas y julio nunca se dispara. Con miles de filas y solo 24 horas distintas,
cada hora tiene unas 190 observaciones: la curva no puede perseguir a ningún punto suelto, solo puede acercarse al promedio de cada hora.
**Aquí el sobreajuste no se nota.**

Para verlo hay que hacer lo que pasa muchas veces en la vida real: tener pocos datos. Usamos **solo 12 horas de un lunes de junio**
(el 22), las mismas del widget de la slide, y seguimos validando con todo julio.

In [ ]:
lunes = train[train.hour.dt.date == pd.Timestamp("2026-06-22").date()]
pocos = lunes[lunes.hour_of_day.isin([0, 1, 3, 4, 5, 7, 8, 11, 14, 17, 19, 21])]
X_pocos, y_pocos = pocos[["hour_of_day"]], pocos.departures
display(pocos.set_index("hour_of_day").departures.to_frame("salidas el lunes 22 de junio").T)

**¿Qué esperamos ver?** Doce números, uno por hora. Faltan horas clave, como las 6, 9 y 18: entre 11 a.m. y 5 p.m. solo hay un punto.

In [ ]:
modelos_pocos = {g: ajusta(g, X_pocos, y_pocos) for g in grados}
brinco_j = np.random.default_rng(1).uniform(-0.3, 0.3, len(julio))


def ver_grado(grado=1):
    m = modelos_pocos[grado]
    mae_tr = mean_absolute_error(y_pocos, m.predict(X_pocos))
    mae_ju = mean_absolute_error(y_julio, m.predict(X_julio))
    plt.figure(figsize=(9, 4))
    plt.scatter(julio.hour_of_day + brinco_j, y_julio, s=8, alpha=0.25, color="gray", edgecolors="none", label="julio (valida)")
    plt.scatter(X_pocos.hour_of_day, y_pocos, s=70, color=COLORES["w"], zorder=3, label="las 12 horas (entrena)")
    plt.plot(horas, m.predict(rejilla), color=COLORES["error"], lw=3, label=f"grado {grado}")
    plt.ylim(-10, 60)
    plt.xlabel("hora del día")
    plt.ylabel("salidas")
    plt.xticks(range(0, 24, 2))
    plt.title(f"Grado {grado}: MAE train {mae_tr:.2f}   MAE julio {mae_ju:.2f}")
    plt.legend(loc="upper left")
    plt.show()


try:
    import ipywidgets as widgets
    widgets.interact(ver_grado, grado=widgets.IntSlider(value=1, min=1, max=12, step=1, description="grado"))
except ImportError:
    print("No hay ipywidgets: usa la celda de abajo.")

In [ ]:
# Si el widget no aparece: cambia el grado (1 a 12) y vuelve a correr esta celda
ver_grado(5)

**¿Qué esperamos ver?** Con grado 1 a 4 la curva apenas se dobla. Con 5 y 6 ya aparece la joroba de la mañana y el MAE de julio baja.
De 7 en adelante la curva pasa cada vez más cerca de los puntos azules y, entre ellos, se sale de la gráfica:
el MAE de train baja y el de julio se dispara. Con 11 o 12 pasa exactamente por los 12 puntos (MAE train 0) y en julio no sirve para nada.

In [ ]:
curva_pocos = pd.DataFrame([metricas(modelos_pocos[g], g, X_pocos, y_pocos) for g in grados]).set_index("modelo")
curva_pocos.index.name = "grado"
# lo que mínimos cuadrados baja de verdad es el error al cuadrado: su raíz (RMSE) en train nunca sube
curva_pocos["RMSE train"] = [np.sqrt(np.mean((modelos_pocos[g].predict(X_pocos) - y_pocos) ** 2)) for g in grados]
display(curva_pocos[["MAE train", "RMSE train", "MAE julio"]].round(2).T)

plt.figure(figsize=(8, 3.8))
plt.plot(grados, curva_pocos["MAE train"], marker="o", label="train (12 horas), MAE")
plt.plot(grados, curva_pocos["RMSE train"], linestyle="--", color=COLORES["b"], label="train (12 horas), RMSE")
plt.plot(grados, curva_pocos["MAE julio"], marker="o", color=COLORES["error"], label="julio")
plt.yscale("symlog", linthresh=10)
plt.xlabel("grado")
plt.ylabel("MAE (escala log arriba de 10)")
plt.xticks(list(grados))
plt.title("Con 12 horas: el error en train baja, el de julio no")
plt.legend()
plt.show()

**¿Qué esperamos ver?** Train va de unas 4 bicis a 0. Julio baja hasta el grado 5 y luego sube a cientos y miles de bicis de error.
Un detalle honesto: de grado 1 a 3 el MAE de train sube un poco (3.83 a 4.07), porque mínimos cuadrados baja el error **al cuadrado**, no el MAE.
La línea punteada es el RMSE de train, la raíz de ese error al cuadrado: esa nunca sube.

Eso es el **sobreajuste**: el modelo aprende los 12 puntos, no el patrón de la estación.

## 5. Escoger el grado con julio

$$d^\ast = \arg\min_{d} \ \text{MAE}_{\text{julio}}(d)$$

En palabras: el grado que escogemos es el que menos se equivoca en julio, el mes que no se usó para ajustar.

**¿Por qué no el de menor error en train?** Porque en train siempre gana el grado más alto, como acabamos de ver. Julio hace de examen.

In [ ]:
mejor_pocos = int(curva_pocos["MAE julio"].idxmin())
mejor_todo = int(curva_todo["MAE julio"].idxmin())
print(f"Con 12 horas, julio escoge el grado {mejor_pocos}")
print(f"Con todo train, julio escoge el grado {mejor_todo}")

fig, (izq, der) = plt.subplots(1, 2, figsize=(10, 3.8))
for eje, curva, mejor, titulo in [(izq, curva_pocos, mejor_pocos, "Con 12 horas"), (der, curva_todo, mejor_todo, "Con todo train")]:
    g_barras = sorted({1, mejor_pocos, mejor_todo, 12})
    vals = curva.loc[g_barras, "MAE julio"]
    etiquetas = ["recta" if g == 1 else f"grado {g}" + (" (el mejor)" if g == mejor else "") for g in g_barras]
    colores_b = [COLORES["error"] if g == 1 else COLORES["yhat"] if g == mejor else COLORES["lambda"] for g in g_barras]
    eje.bar(etiquetas, vals, color=colores_b, edgecolor=COLORES["tinta"])
    for k, v in enumerate(vals):
        eje.annotate(f"{v:,.2f}", (k, v), xytext=(0, 4), textcoords="offset points", ha="center", fontweight="bold")
    eje.set(title=titulo, ylabel="MAE en julio")
izq.set_yscale("log")
izq.set_ylabel("MAE en julio (escala log)")
plt.tight_layout()
plt.show()

**¿Qué esperamos ver?** Con 12 horas gana el grado 5 (MAE en julio de 6.4 contra 8.6 de la recta) y el grado 12 es un desastre (casi 2 mil bicis de error).
Con todo train el grado 5 ya no gana: gana el 12 (5.2 contra 6.7 del grado 5 y 8.5 de la recta). El mismo grado es el peor o el mejor según **cuántos datos** tengamos.
Por eso el grado no se escoge de una vez y para siempre: se escoge con validación, cada vez.

Volvamos a la gráfica del diagnóstico con el mejor modelo de todo train. ¿Se le quitó la forma a los errores?

In [ ]:
mejor = ajusta(mejor_todo, X_train, y_train)
err_mejor = pd.DataFrame({"train": errores_por_hora(mejor, train, X_train),
                          "julio": errores_por_hora(mejor, julio, X_julio)})

fig, (izq, der) = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for eje, col in [(izq, "train"), (der, "julio")]:
    eje.bar(err_recta.index - 0.2, err_recta[col], width=0.4, color=COLORES["error"], edgecolor=COLORES["tinta"], label="recta")
    eje.bar(err_mejor.index + 0.2, err_mejor[col], width=0.4, color=COLORES["yhat"], edgecolor=COLORES["tinta"], label=f"grado {mejor_todo}")
    eje.axhline(0, color=COLORES["tinta"], lw=1.5)
    eje.set(title=f"Errores promedio por hora en {col}", xlabel="hora del día", xticks=range(0, 24, 2))
izq.set_ylabel("error promedio ŷ − y (bicis)")
izq.legend()
plt.tight_layout()
plt.show()

display(pd.DataFrame({"recta": err_recta.abs().mean(), f"grado {mejor_todo}": err_mejor.abs().mean()},
                     ).rename_axis("error promedio por hora, en valor absoluto").round(2))
print("Salidas promedio a las 7, 8 y 18 h:")
display(pd.DataFrame({"train": prom, "julio": y_julio.groupby(julio.hour_of_day).mean()}).loc[[7, 8, 18]].round(1).T)

**¿Qué esperamos ver?** En train las barras verdes se quedan cerca de cero en todas las horas: la forma desapareció
(en promedio, de unas 5.5 bicis por hora a menos de 2). En julio también se achican, pero quedan negativas en las horas pico:
julio tuvo más viajes a las 7, 8 y 18 h que el promedio de octubre a junio. Eso ya no es curva que le falte al modelo,
es que julio fue un mes más movido; la hora sola no lo puede saber.

## 6. Lo que un analista cuida

### a) Fuera del rango, el polinomio inventa

La hora va de 0 a 23. ¿Qué dice la curva si le preguntamos por la hora 26 (las 2 a.m. del día siguiente) o la 30?

In [ ]:
lejos = np.linspace(-3, 30, 400)
rejilla_lejos = pd.DataFrame({"hour_of_day": lejos})
g5 = ajusta(5, X_train, y_train)

plt.figure(figsize=(9, 4))
plt.axvspan(-3, 0, color="#eeeeee")
plt.axvspan(23, 30, color="#eeeeee", label="fuera de los datos")
plt.plot(prom.index, prom.values, color=COLORES["tinta"], marker="o", label="promedio de cada hora")
plt.plot(lejos, recta.predict(rejilla_lejos), color=COLORES["error"], label="recta")
plt.plot(lejos, g5.predict(rejilla_lejos), color=COLORES["lambda"], label="grado 5")
plt.plot(lejos, mejor.predict(rejilla_lejos), color=COLORES["yhat"], label=f"grado {mejor_todo}")
plt.ylim(-60, 80)
plt.xlabel("hora (más de 23 es el día siguiente)")
plt.ylabel("salidas predichas")
plt.title("Dentro de 0 a 23 las curvas siguen los datos; afuera, inventan")
plt.legend(loc="lower left", fontsize=9)
plt.show()

preguntas = pd.DataFrame({"hour_of_day": [24, 26, 30]})
display(pd.DataFrame({"recta": recta.predict(preguntas), "grado 5": g5.predict(preguntas),
                      f"grado {mejor_todo}": mejor.predict(preguntas)},
                     index=["hora 24 (medianoche)", "hora 26 (2 a.m.)", "hora 30 (6 a.m.)"]).round(1))

**¿Qué esperamos ver?** Entre 0 y 23 los dos polinomios siguen el promedio. Afuera se van a números absurdos:
el grado 5 predice salidas negativas a las 2 a.m. y el grado más alto cientos de miles de bicis negativas a las 6 a.m.
La realidad es que a las 2 a.m. sale casi nada y a las 6 a.m. salen unas 12. Un polinomio solo describe donde hay datos.
La recta también se equivoca afuera, solo que más despacio.

### b) Con pocos datos, el grado alto se dispara entre los puntos

Ya lo vimos en el widget. Aquí, el valor más grande que alcanza cada curva **dentro** del día, ajustada con las 12 horas:

In [ ]:
maximos = pd.Series({g: np.abs(modelos_pocos[g].predict(rejilla)).max() for g in grados}, name="|ŷ| más grande entre 0 y 23 h")
maximos.index.name = "grado"
display(maximos.round(0).to_frame().T)

plt.figure(figsize=(9, 4))
plt.scatter(X_pocos.hour_of_day, y_pocos, s=70, color=COLORES["w"], zorder=3, label="las 12 horas")
for g, color in [(5, COLORES["yhat"]), (8, COLORES["lambda"]), (10, COLORES["error"])]:
    plt.plot(horas, modelos_pocos[g].predict(rejilla), color=color, label=f"grado {g}")
plt.ylim(-60, 100)
plt.xlabel("hora del día")
plt.ylabel("salidas predichas")
plt.xticks(range(0, 24, 2))
plt.legend(loc="upper left")
plt.show()

**¿Qué esperamos ver?** Hasta el grado 6 la curva no pasa de unas 40 bicis, que es razonable para esta estación.
Con grado 8 ya llega a más de 200 y con 10 a más de 10 mil, entre dos horas que sí tenemos. Ninguna estación saca 10 mil bicis en una hora.
Si una curva predice algo imposible entre los datos, el grado está alto para los datos que hay.

### c) Otra salida: la hora como categoría

En lugar de potencias, una columna por hora: `hora_0`, `hora_1`, ..., `hora_23`. Cada fila tiene un 1 en la columna de su hora y 0 en las demás.

$$\hat y = w_0\,[x = 0] + w_1\,[x = 1] + \dots + w_{23}\,[x = 23]$$

En palabras: $[x = h]$ vale 1 si la fila es de la hora $h$ y 0 si no, así que cada hora recibe su propio número $w_h$, sin obligarla
a seguir una curva. Es lo que hace `OneHotEncoder`.

**¿Por qué probarla?** Porque no tenemos que escoger grado y no se puede disparar entre horas: no hay "entre horas".
A cambio, no sabe nada de una hora que no vio, y necesita datos de cada una de las 24.

In [ ]:
uno_caliente = OneHotEncoder(sparse_output=False).fit(X_train)
muestra = pd.DataFrame(uno_caliente.transform(unas_horas), index=unas_horas.hour_of_day,
                       columns=[f"hora_{h}" for h in range(24)]).astype(int)
display(muestra.iloc[:, [0, 1, 2, 6, 7, 8, 12, 18, 23]])

hora_cat = make_pipeline(OneHotEncoder(handle_unknown="ignore"), LinearRegression()).fit(X_train, y_train)
resumen = pd.DataFrame([metricas(recta, "recta (2 números)"),
                        metricas(mejor, f"grado {mejor_todo} ({mejor_todo + 1} números)"),
                        metricas(hora_cat, "hora como categoría (24 números)")]).round(3)
display(resumen)

plt.figure(figsize=(9, 4))
plt.plot(prom.index, prom.values, color=COLORES["tinta"], marker="o", lw=1, label="promedio de cada hora")
plt.plot(horas, recta.predict(rejilla), color=COLORES["error"], label="recta")
plt.plot(horas, mejor.predict(rejilla), color=COLORES["yhat"], label=f"grado {mejor_todo}")
plt.step(range(24), hora_cat.predict(pd.DataFrame({"hour_of_day": range(24)})), where="mid",
         color=COLORES["lambda"], label="hora como categoría")
plt.xlabel("hora del día")
plt.ylabel("salidas predichas")
plt.xticks(range(0, 24, 2))
plt.legend(loc="upper left")
plt.show()

**¿Qué esperamos ver?** En la tabla de columnas, un solo 1 por fila. En el resumen, la hora como categoría gana: MAE en julio de unos 4.7,
contra 5.2 del mejor polinomio y 8.5 de la recta. En la gráfica, sus escalones caen justo en el promedio de cada hora,
porque eso es lo que aprende: el promedio de train de esa hora.

**El puente a S2:** `x²`, `x³` y `hora_7` son **características derivadas**: columnas que inventamos a partir de otra.
Escoger cuáles inventar es casi siempre lo que más mueve un modelo, y mañana lo hacemos con más columnas a la vez.

## Tus preguntas y reflexiones

No se entregan. Escribe dos o tres líneas en cada una; las retomamos al inicio de S2.

**1. ¿Qué error le cuesta más a MiBici a las 7 a.m.?** Aun con el mejor polinomio, en julio el modelo se quedó corto en las horas pico.
Si redistribución usa esta predicción para decidir cuántas bicis dejar en la estación 51, ¿qué pasa con los usuarios cuando el modelo se queda corto?
¿Y cuando se pasa? ¿Cuál preferirías y qué le pedirías al modelo?

*Tip: regresa a la gráfica de errores por hora en julio y fíjate en el signo.*

_Escribe aquí._

**2. MiBici amplía el horario hasta la 1 a.m.** Te piden la predicción para esas horas nuevas con la curva que ya tienes. ¿Le crees?
¿Qué harías antes de entregar un número?

*Tip: la gráfica de 0 a 30 horas.*

_Escribe aquí._

**3. Abren una estación nueva y solo tiene dos semanas de historia.** Con 12 horas ganó el grado 5; con nueve meses ganó el 12 y la hora como
categoría les ganó a todos. ¿Qué modelo usarías para la estación nueva y por qué? ¿Cuándo cambiarías de modelo?

*Tip: piensa cuántos datos hay por cada número que el modelo tiene que aprender.*

_Escribe aquí._